# Data Processing

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("ship_movement_data_with_precipitation.csv")

In [3]:
num_high_severity = (data['Storm_Severity'] == 'Low').sum()

# Data Understanding

In [4]:
data.head()

,Ship_Type,Ship_Size,Fuel_Level_tonnes,Ship_Load_%,Wind_Speed_knots,Precipitation_mm_hr,Wave_Height_m,Storm_Severity,Distance_to_Disaster_km,Adjusted_Speed_knots
0,Cargo,Medium,53.26,75.0,14.95,36.50,7.14,Medium,149.46,6.90
1,Tanker,Small,48.46,98.0,13.32,9.23,4.24,Low,47.41,5.00
2,Passenger,Small,45.06,89.0,7.05,17.33,4.29,Low,63.18,5.30
3,Cargo,Large,137.36,84.0,24.29,33.16,6.21,Medium,90.34,8.80
4,Container,Large,197.99,66.0,19.06,24.10,7.59,Medium,101.83,9.22


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Ship_Type                10000 non-null  object 
 1   Ship_Size                10000 non-null  object 
 2   Fuel_Level_tonnes        10000 non-null  float64
 3   Ship_Load_%              10000 non-null  float64
 4   Wind_Speed_knots         10000 non-null  float64
 5   Precipitation_mm_hr      10000 non-null  float64
 6   Wave_Height_m            10000 non-null  float64
 7   Storm_Severity           10000 non-null  object 
 8   Distance_to_Disaster_km  10000 non-null  float64
 9   Adjusted_Speed_knots     10000 non-null  float64
dtypes: float64(7), object(3)
memory usage: 781.4+ KB


In [6]:
data.describe()

,Fuel_Level_tonnes,Ship_Load_%,Wind_Speed_knots,Precipitation_mm_hr,Wave_Height_m,Distance_to_Disaster_km,Adjusted_Speed_knots
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,86.262148,79.765700,20.181239,25.002492,7.265648,248.431907,7.848698
std,51.297952,11.511023,11.571738,14.338665,4.394308,144.642050,4.702625
min,20.000000,60.000000,0.010000,0.000000,0.160000,0.010000,0.000000
25%,42.570000,70.000000,10.160000,12.690000,3.810000,122.132500,4.300000
50%,74.035000,80.000000,20.235000,25.100000,6.350000,246.855000,7.610000
75%,123.487500,90.000000,30.260000,37.232500,10.060000,375.242500,11.090000
max,199.990000,100.000000,40.000000,50.000000,21.490000,499.990000,19.470000


# Data Manipulation

### Applying StandardScaler for Feature Scaling

In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [8]:
# Normalize/Scale numerical features
# Define numerical columns to scale
numerical_cols = ['Fuel_Level_tonnes', 'Ship_Load_%', 'Wind_Speed_knots', 
                  'Precipitation_mm_hr', 'Wave_Height_m', 'Distance_to_Disaster_km']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to numerical columns
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

### Applying OneHotEncoder for Feature encoding

In [9]:
# Encode categorical features
# Define categorical columns to encode
categorical_cols = ['Ship_Type', 'Ship_Size', 'Storm_Severity']

# Initialize OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)  # Drop first to avoid multicollinearity

# Apply OneHotEncoder to categorical columns
encoded_features = encoder.fit_transform(data[categorical_cols])

# Create DataFrame for encoded features
encoded_columns = encoder.get_feature_names_out(categorical_cols)
encoded_data = pd.DataFrame(encoded_features, columns=encoded_columns)

# Concatenate the original DataFrame with encoded features and drop original categorical columns
data = pd.concat([data.reset_index(drop=True), encoded_data.reset_index(drop=True)], axis=1)
data.drop(columns=categorical_cols, inplace=True)

In [10]:
print(encoded_columns)

['Ship_Type_Cargo' 'Ship_Type_Container' 'Ship_Type_Passenger'
 'Ship_Type_Tanker' 'Ship_Size_Large' 'Ship_Size_Medium' 'Ship_Size_Small'
 'Storm_Severity_High' 'Storm_Severity_Low' 'Storm_Severity_Medium']


### Dividing Data into training and testing set

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
# Split the data into training, validation, and testing sets
# Define target variable and features
target = 'Adjusted_Speed_knots'
X = data.drop(columns=[target])
y = data[target]

# Split the data into train (70%), validation (15%), and test (15%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Print the shapes of the resulting sets
print("Training set shape: ", X_train.shape, y_train.shape)
print("Testing set shape: ", X_test.shape, y_test.shape)

Training set shape:  (7500, 16) (7500,)
Testing set shape:  (2500, 16) (2500,)


In [13]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7500 entries, 4901 to 7270
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Fuel_Level_tonnes        7500 non-null   float64
 1   Ship_Load_%              7500 non-null   float64
 2   Wind_Speed_knots         7500 non-null   float64
 3   Precipitation_mm_hr      7500 non-null   float64
 4   Wave_Height_m            7500 non-null   float64
 5   Distance_to_Disaster_km  7500 non-null   float64
 6   Ship_Type_Cargo          7500 non-null   float64
 7   Ship_Type_Container      7500 non-null   float64
 8   Ship_Type_Passenger      7500 non-null   float64
 9   Ship_Type_Tanker         7500 non-null   float64
 10  Ship_Size_Large          7500 non-null   float64
 11  Ship_Size_Medium         7500 non-null   float64
 12  Ship_Size_Small          7500 non-null   float64
 13  Storm_Severity_High      7500 non-null   float64
 14  Storm_Severity_Low       7

# Creating ANN Model

In [14]:
import tensorflow as tf

2024-12-03 18:32:26.076848: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-03 18:32:26.104045: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [15]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

In [16]:
model = Sequential()

In [17]:
model.add(Input(shape = (16,)))
model.add(Dense(units = 32, activation='relu'))
model.add(Dense(units = 64, activation='relu'))
model.add(Dense(units = 32, activation='relu'))
model.add(Dense(units = 8, activation='relu'))
model.add(Dense(units = 1, activation='relu'))

In [18]:
model.compile(optimizer='adam', loss="mean_absolute_error", metrics='mean_absolute_error')

In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                544       
                                                                 
 dense_1 (Dense)             (None, 64)                2112      
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 8)                 264       
                                                                 
 dense_4 (Dense)             (None, 1)                 9         
                                                                 
Total params: 5009 (19.57 KB)
Trainable params: 5009 (19.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
model.fit(X_train, y_train, epochs=20)

Epoch 1/20
235/235 [==============================] - 1s 937us/step - loss: 2.2659 - mean_absolute_error: 2.2659
Epoch 2/20
235/235 [==============================] - 0s 992us/step - loss: 0.3739 - mean_absolute_error: 0.3739
Epoch 3/20
235/235 [==============================] - 0s 973us/step - loss: 0.1719 - mean_absolute_error: 0.1719
Epoch 4/20
235/235 [==============================] - 0s 958us/step - loss: 0.1283 - mean_absolute_error: 0.1283
Epoch 5/20
235/235 [==============================] - 0s 942us/step - loss: 0.1101 - mean_absolute_error: 0.1101
Epoch 6/20
235/235 [==============================] - 0s 927us/step - loss: 0.0916 - mean_absolute_error: 0.0916
Epoch 7/20
235/235 [==============================] - 0s 889us/step - loss: 0.0950 - mean_absolute_error: 0.0950
Epoch 8/20
235/235 [==============================] - 0s 1ms/step - loss: 0.0731 - mean_absolute_error: 0.0731
Epoch 9/20
235/235 [==============================] - 0s 1ms/step - loss: 0.0724 - mean_absolute_e

In [22]:
y_pred = model.predict(X_test)

79/79 [==============================] - 0s 631us/step


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [24]:
abs_error = mean_absolute_error(y_test, y_pred)
sqr_error = mean_squared_error(y_test, y_pred)
rms_error = root_mean_squared_error(y_test, y_pred)

In [25]:
print("mean_absolute_error(y_test, y_pred)", abs_error)
print("mean_squared_error(y_test, y_pred)", sqr_error)
print("root_mean_squared_error(y_test, y_pred)", rms_error)

mean_absolute_error(y_test, y_pred) 0.04133723966026306
mean_squared_error(y_test, y_pred) 0.00309704852612256
root_mean_squared_error(y_test, y_pred) 0.05565113229865642


In [26]:
import joblib

# Assuming scaler, encoder, and model are already trained
pipeline = {
    'scaler': scaler,          # StandardScaler object
    'encoder': encoder,        # OneHotEncoder object
    'model': model             # Trained ANN model (e.g., from TensorFlow or PyTorch)
}

# Save the pipeline to a file
joblib.dump(pipeline, 'ship_speed_pipeline.pkl')
print("Pipeline saved to 'ship_pipeline.pkl'")

Pipeline saved to 'ship_pipeline.pkl'


# Inferencing the model

In [27]:
import joblib
import pandas as pd

In [28]:
import time

In [29]:
# Load the pipeline from the file
loaded_pipeline = joblib.load('ship_speed_pipeline.pkl')

# Extract components
scaler = loaded_pipeline['scaler']
encoder = loaded_pipeline['encoder']
model = loaded_pipeline['model']

print("Pipeline loaded successfully")

Pipeline loaded successfully


In [30]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                544       
                                                                 
 dense_1 (Dense)             (None, 64)                2112      
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 8)                 264       
                                                                 
 dense_4 (Dense)             (None, 1)                 9         
                                                                 
Total params: 5009 (19.57 KB)
Trainable params: 5009 (19.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
# Storm Severity: Based on a combination of wind speed, wave height, and precipitation
def determine_storm_severity(row):
    severity_score = row['Wind_Speed_knots'] + row['Wave_Height_m'] * 5 + row['Precipitation_mm_hr'] * 0.5
    print("severity_score : ", severity_score)
    if severity_score > 10:
        return 'High'
    elif severity_score > 5:
        return 'Medium'
    else:
        return 'Low'

In [38]:
def predict(inputs):
    start = time.time_ns()
    # Wrap scalar values in lists
    inputs = {key: [value] for key, value in inputs.items()}
    
    # Converting JSON object into dataframe
    df = pd.DataFrame.from_dict(inputs)

    # print(df)
    
    # Select numerical features for scaling
    numerical_features = ["Fuel_Level_tonnes", "Ship_Load_%", "Wind_Speed_knots", 
                          "Precipitation_mm_hr", "Wave_Height_m", "Distance_to_Disaster_km"]
    
    # Scale numerical features
    df[numerical_features] = scaler.transform(df[numerical_features])

    # Detect Storm Severity level
    df['Storm_Severity'] = df.apply(determine_storm_severity, axis=1)
    
    # Encode categorical features
    categorical_features = ["Ship_Type", "Ship_Size", "Storm_Severity"]
    encoded_data = encoder.transform(df[categorical_features])
    encoded_columns = encoder.get_feature_names_out(categorical_features)
    
    # Create a DataFrame from the encoded features
    encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=df.index)
    
    # Combine scaled numerical features and encoded categorical features
    processed_df = pd.concat([df[numerical_features], encoded_df], axis=1)
    
    # Reorder columns to match the required format
    required_columns = [
        "Fuel_Level_tonnes", "Ship_Load_%", "Wind_Speed_knots", 
        "Precipitation_mm_hr", "Wave_Height_m", "Distance_to_Disaster_km",
        "Ship_Type_Cargo", "Ship_Type_Container", "Ship_Type_Passenger", "Ship_Type_Tanker",
        "Ship_Size_Large", "Ship_Size_Medium", "Ship_Size_Small",
        "Storm_Severity_High", "Storm_Severity_Low", "Storm_Severity_Medium"
    ]
    processed_df = processed_df[required_columns]

    result = model.predict(processed_df)
    end = time.time_ns()
    print("Total time taken : ", end-start)
    return result[0][0]

In [45]:
data = {
    "Ship_Type": "Tanker",
    "Ship_Size": "Large",
    "Fuel_Level_tonnes": 131.71,
    "Ship_Load_%": 84,
    "Wind_Speed_knots": 3.52,
    "Precipitation_mm_hr": 35.48,
    "Wave_Height_m": 0.8,
    "Distance_to_Disaster_km": 254.79,
}

In [46]:
predict(data)

severity_score :  -8.431728448913228
1/1 [==============================] - 0s 13ms/step
Total time taken :  49920509


16.41017

In [7]:
import requests

In [8]:
response = requests.get("http://0.0.0.0:8000/predict_speed", json = data)

ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=8000): Max retries exceeded with url: /predict_speed (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x77dc744cae00>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [5]:
print(response.json())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)